# 2.2 Transformer 简介

2.3 节将钻进 Attention 的内部，2.4 节分析它为什么慢，2.5 节引出 FlashAttention。但在钻进细节之前，值得先坐直升机看一眼整栋建筑——**Transformer**：Attention 在其中的位置、它与 FFN 的分工、以及为什么序列越长，Attention 越值得专门优化。

一句话定位：**FlashAttention 优化的是 Transformer 里唯一随序列长度平方增长的部件**。看懂全局，才知道我们造的轮子安在哪里。

## 一、Transformer 要解决什么问题

2017 年之前，序列建模的主流是 RNN / LSTM：逐个 token 处理，第 t 步依赖第 t−1 步的隐状态。两大痛点：

1. **串行**：必须算完第 t−1 步才能算第 t 步，无法并行训练——GPU 上万核心大量闲置；
2. **长程依赖衰减**：信息要经过 t 步接力传递，隔得太远就“传丢了”。

Transformer 的答案（论文 *Attention Is All You Need*, 2017）：**自注意力**——任意两个 token 之间直接建立一条连接，一步到位；整个序列的所有位置并行计算。

| 维度 | RNN / LSTM | 自注意力 |
|--|--|--|
| 并行性 | 天然串行 | 全序列并行 |
| 任意两 token 的信息路径长度 | O(n) | O(1) |
| 单层计算复杂度 | O(n·d²) | **O(n²·d)** |

注意最后一行：自注意力用 **O(n²)** 的计算代价，换来了并行性与全局视野。这个 n²，就是本课程整本书要对付的“敌人”——FlashAttention 并不消灭它（理论计算量几乎不变），而是通过重构数据流，把它背后的**访存**开销压掉。

## 二、整体架构：原版 Transformer（2017，Encoder-Decoder）

![Transformer 架构](images/transformer_architecture.svg)

对照左图，自下而上：

| 组件 | 作用 | 一句话备注 |
|--|--|--|
| Embedding | 把 token id 变成 d_model 维向量 | 查表操作 |
| 位置编码 | 给每个位置注入信号 | Attention 本身对顺序不敏感（排列不变），必须额外提供位置信息 |
| Encoder（×N） | Self-Attention + FFN | 每个词看到所有词 |
| Decoder 的 Masked Self-Attention | 只许看已生成的词 | “不许偷看未来”——2.5 节因果掩码块跳过的原型 |
| Cross-Attention | Q 来自译文，K/V 来自原文 | 翻译任务的桥 |
| Linear + Softmax | 对词表打分 | 预测下一个词的概率 |

两个值得记住的细节：

- **Attention 在原版里出现三次**（encoder self / decoder masked self / cross），但形态完全相同——都是“打分 → 归一化 → 加权求和”三步（2.3 节展开）；
- **位置编码从绝对走向相对**：原版用固定正弦编码；现代大模型几乎都用 **RoPE**（旋转位置编码）——把位置信息“旋转”进向量。记住这个名字，2.9 节讲 MLA 时它还会回来：RoPE 与低秩压缩不兼容，正是 MLA 结构复杂的根源。

## 三、现代大模型：Decoder-Only 一统天下

GPT 系列证明了一件事：**只要保留 Decoder 的“预测下一个词”目标，把规模堆上去，通用能力就会涌现**。于是今天的开源大模型（Llama、DeepSeek、Qwen、GLM……）全部是右图的结构：

> token → Embedding（+RoPE）→ **[Masked Self-Attention → FFN] × N 层** → LM Head → 下一个 token

原版 → 现代的演化清单：

| 项目 | 2017 原版 | 现代 LLM |
|--|--|--|
| 框架 | Encoder-Decoder | **Decoder-Only** |
| 归一化 | LayerNorm | RMSNorm（更省） |
| 位置编码 | 正弦绝对编码 | RoPE |
| FFN | ReLU 两矩阵 | SwiGLU 三矩阵 / MoE |
| 规模 | 65M | 7B ~ 700B（靠层数与宽度堆叠） |

注意右图里那个自回归循环箭头：**推理就是“预测一个 token、拼回输入、再预测下一个”的循环**。这个循环的两种运行节奏，直接决定了 FA 算子的两条路径（见第五节）。

## 四、解剖一个 Decoder Block：Attention 与 FFN 的分工

![Decoder Block 数据流](images/transformer_block_dataflow.svg)

一个 block 内，数据依次流过：

1. **QKV 投影**：x（n × d_model）右乘三个投影矩阵，得到 Q、K、V；
2. **Attention**：2.3 节的三步计算，输出仍是 n × d_model；
3. **残差 + RMSNorm**：稳定训练的标配；
4. **FFN**：SwiGLU——升维约 4 倍、门控激活、再降回来；
5. **残差 + RMSNorm** → 送入下一层。

先看**参数量**账单：

In [1]:
def block_param_count(d_model, d_ff, use_swiglu=True):
    attn = 4 * d_model ** 2                    # W_Q、W_K、W_V、W_O 四个投影
    ffn = 3 * d_model * d_ff if use_swiglu else 2 * d_model * d_ff
    norm = 4 * d_model                         # 两个 RMSNorm，各一个增益向量
    return {'Attention投影': attn, 'FFN': ffn, 'Norm等小项': norm}

cfgs = [('Llama-3-8B 单块近似', 4096, 14336),
        ('Llama-2-7B 单块近似', 4096, 11008)]
for name, dm, dff in cfgs:
    p = block_param_count(dm, dff)
    total = sum(p.values())
    print(f'{name}：d_model={dm}, d_ff={dff}')
    for k, v in p.items():
        print(f'   {k:10s} {v/1e6:9.1f} M（{v/total:5.1%}）')
    print()

Llama-3-8B 单块近似：d_model=4096, d_ff=14336
   Attention投影      67.1 M（27.6%）
   FFN            176.2 M（72.4%）
   Norm等小项          0.0 M（ 0.0%）

Llama-2-7B 单块近似：d_model=4096, d_ff=11008
   Attention投影      67.1 M（33.2%）
   FFN            135.3 M（66.8%）
   Norm等小项          0.0 M（ 0.0%）



结论：**单块参数约 2/3 在 FFN，Attention 投影只占约 1/3**。那为什么值得为 Attention 专门写一个算子（甚至开一门课）？因为决定“值得优化”的不是参数量，而是**计算量随序列长度的增长方式**——再看计算量账单：

In [1]:
def block_flops(n, d_model, d_ff, causal=True):
    proj = 4 * n * d_model ** 2                       # QKVO 投影：随 n 线性
    factor = 0.5 if causal else 1.0
    attn_core = factor * 4 * n * n * d_model          # QK^T 与 PV 两个 GEMM：随 n 平方
    ffn = 6 * n * d_model * d_ff                      # SwiGLU 三个 GEMM：随 n 线性
    return {'QKVO投影': proj, 'Attention核心': attn_core, 'FFN': ffn}

dm, dff = 4096, 14336
print(f'd_model={dm}, d_ff={dff}，开启因果掩码（对角减半）')
for n in [1024, 4096, 16384, 65536, 131072]:
    f = block_flops(n, dm, dff)
    total = sum(f.values())
    print(f'n = {n:6d}：Attention 核心 FLOPs 占比 = {f["Attention核心"]/total:6.1%}')

d_model=4096, d_ff=14336，开启因果掩码（对角减半）
n =   1024：Attention 核心 FLOPs 占比 =   2.0%
n =   4096：Attention 核心 FLOPs 占比 =   7.4%
n =  16384：Attention 核心 FLOPs 占比 =  24.2%
n =  65536：Attention 核心 FLOPs 占比 =  56.1%
n = 131072：Attention 核心 FLOPs 占比 =  71.9%

FFN 与投影都是 n 的**线性**函数，只有 Attention 核心是 **n²**：4k 序列时它只占 7%，128k 时超过 70% 成为第一瓶颈。长上下文时代，“把 Attention 算快”就是把最贵的那段代码算快——这就是 FlashAttention（以及本课程）存在的理由。

## 五、一次推理的一生：Prefill 与 Decode

同一条 block 数据流，在推理时以两种节奏运行（见上图右下角）：

| 阶段 | 每步输入 | Q 的形状 | 瓶颈体质 |
|--|--|--|--|
| **Prefill**（处理整段 prompt） | n 个 token | n × d | 大矩阵 GEMM，**算力瓶颈** |
| **Decode**（逐字生成） | 1 个 token | 1 × d | 每步读全部历史 KV，**带宽瓶颈** |

这正是昇腾 `FusedInferAttentionScore` 算子里 PromptFlashAttention / IncreFlashAttention 两条分支的由来：**同一个数学，两种负载**，kernel 的 Tiling 与分核策略完全不同（第 5 章）。

## 六、动手：一个能跑的最小 Decoder（numpy）

下面用不到 50 行 numpy 搭一个“麻雀虽小五脏俱全”的 decoder-only 模型：2 层、4 头、SwiGLU FFN、RMSNorm、因果掩码。权重随机（它不智能，但结构是真的）：

In [3]:
import numpy as np
np.random.seed(42)

def silu(x):
    return x / (1 + np.exp(-x))

def rms_norm(x):
    return x / np.sqrt((x ** 2).mean(-1, keepdims=True) + 1e-6)

def causal_attention(Q, K, V):
    S = Q @ K.T / np.sqrt(Q.shape[-1])
    S[np.triu_indices(S.shape[0], 1)] = -np.inf      # 因果掩码：不许看未来
    P = np.exp(S - S.max(axis=1, keepdims=True))
    P = P / P.sum(axis=1, keepdims=True)
    return P @ V

# 一个 decoder block：Masked MHA → 残差+Norm → SwiGLU FFN → 残差+Norm
class Block:
    def __init__(self, dm, nh, dff):
        r = lambda a, b: np.random.randn(a, b) * 0.05
        self.nh = nh
        self.Wqkv = r(dm, 3 * dm)          # Q/K/V 三个投影并成一个矩阵
        self.Wo = r(dm, dm)
        self.Wg_u = r(dm, 2 * dff)         # gate 与 up 并成一个矩阵
        self.Wd = r(dff, dm)

    def __call__(self, x):
        n, dm = x.shape
        Q, K, V = np.split(x @ self.Wqkv, 3, axis=-1)
        heads = []
        for i in range(self.nh):           # 多头：按最后一维切份，各算各的
            s, e = i * dm // self.nh, (i + 1) * dm // self.nh
            heads.append(causal_attention(Q[:, s:e], K[:, s:e], V[:, s:e]))
        x = rms_norm(x + np.concatenate(heads, -1) @ self.Wo)
        g, u = np.split(x @ self.Wg_u, 2, axis=-1)
        return rms_norm(x + (silu(g) * u) @ self.Wd)

dm, nh, dff, n_layers, vocab = 64, 4, 128, 2, 100
emb = np.random.randn(vocab, dm) * 0.1
blocks = [Block(dm, nh, dff) for _ in range(n_layers)]
W_head = np.random.randn(dm, vocab) * 0.1

tokens = [3, 17, 42]                        # 一句 prompt：3 个 token
for step in range(3):                       # 自回归：每步预测下一个 token
    x = emb[tokens]
    for blk in blocks:
        x = blk(x)
    nxt = int(np.argmax(x[-1] @ W_head))
    tokens.append(nxt)
    print(f'第 {step+1} 步：输入 {len(tokens)-1} 个 token → 预测下一个 token = {nxt}')

第 1 步：输入 3 个 token → 预测下一个 token = 61
第 2 步：输入 4 个 token → 预测下一个 token = 51
第 3 步：输入 5 个 token → 预测下一个 token = 1

三个观察：

1. **每一层、每个头的 Attention 都完整执行了“打分 → 归一化 → 加权”三步**——2.3 节的全部内容；
2. 本例每步都**重算全部历史 token 的 K/V**（`x @ self.Wqkv` 作用于整个序列）——真实系统用 **KV Cache**：历史 K/V 只算一次、缓存复用，每步只算新 token（2.3 节第七节）；
3. 因果掩码 `S[上三角] = -inf` 保证第 t 个 token 看不到 t 之后——它就是 2.5 节“因果掩码整块跳过”的逐元素版本。

## 七、小结

1. Transformer 用自注意力换并行性与全局视野，代价是 **O(n²·d)** 的计算与中间矩阵；
2. 参数大头在 FFN，但**计算量随 n 的增长由 Attention 主导**——长上下文时它是第一瓶颈；
3. Decoder-Only + 自回归是现代 LLM 的统一形态；推理分 Prefill（算力瓶颈）与 Decode（带宽瓶颈）两种体质；
4. RoPE 位置编码记住即可——2.9 节 MLA 里它会制造大麻烦。

## 章节测验

1. RNN 的两大痛点是什么？Transformer 分别用什么解决？
2. 自注意力的单层计算复杂度是多少？它换来了哪两个好处？
3.（填空）单块参数中 FFN 约占 ____；但序列长度 n 增大时，计算量占比增长最快的是 ____。
4.（判断）n=128k 时 Attention 核心 FLOPs 占比超过 70%，所以应该优先优化 FFN。（对/错）
5. Prefill 与 Decode 各是什么瓶颈？对应 FA 算子的哪两条路径？
6. 为什么需要位置编码？（提示：Attention 对输入顺序是什么性质？）

> 答案见 `answer/02.02_answer.txt`。

下一节：[2.3 Attention 机制基础](02.03_attention_basics.ipynb)——现在钻进那个粉色方块，看看“打分→归一化→加权求和”到底在算什么。